# Workflow Optimizer — V0

**Goal.** Given a *task* (a seed prompt, and optionally a labeled dataset) and a *budget*, automatically find the LLM workflow that maximizes accuracy under a cost ceiling (or minimizes cost above an accuracy floor).

**How it works.** A "workflow" is just a Python function `solve(question, llm) -> answer`. Because it's ordinary code, it can be *any* strategy — one call, chain-of-thought, self-consistency, decomposition, a cheap→expensive router. We:

1. **Profile the task** — infer a description and the answer format (how to *extract* the answer, how to *check* it), and generate a dataset if you didn't provide one.
2. **Design workflows** — a Claude Code agent writes and self-tests several candidate `solve` programs.
3. **Search** — run each program on a held-out test split through one metered `llm()` call site, then pick the best point on the accuracy/cost **Pareto frontier**.

**Setup.** Every cell makes real Anthropic API calls, so set `ANTHROPIC_API_KEY` first. Each workflow runs under a per-query budget cap (max model calls + tokens); a program that overspends or crashes simply scores 0.

In [ ]:
import os, re, json, statistics

import anthropic
import pandas as pd
import matplotlib.pyplot as plt

# ---- Pricing: USD per 1,000,000 tokens (input, output) --------------------
PRICES = {
    "claude-haiku-4-5": (1.0,  5.0),
    "claude-sonnet-5":  (3.0, 15.0),
    "claude-opus-4-8":  (5.0, 25.0),
}

def cost_usd(model, tokens_in, tokens_out):
    pin, pout = PRICES[model]
    return (tokens_in * pin + tokens_out * pout) / 1_000_000

# ---- Anthropic API client -------------------------------------------------
# Every rollout is a real API call, so a key is required.
if not os.environ.get("ANTHROPIC_API_KEY"):
    raise RuntimeError("Set ANTHROPIC_API_KEY in your shell env before running.")
client = anthropic.Anthropic()

print("Anthropic client ready.")

In [ ]:
# ---- Define the problem (the ONLY per-task input) -------------------------
# Describe the task in SEED_PROMPT. Optionally supply your own labeled DATASET
# as a list of {"question": <input str>, "answer": <target>}; leave it None to
# have one generated. The answer format (numeric / exact-match / free-form) is
# figured out automatically in the profiling step below.
SEED_PROMPT = ("Solve multi-step grade-school math word problems. "
               "Each answer is a single integer.")
DATASET = None      # or: [{"question": "...", "answer": 42}, ...]

print("Seed prompt set." +
      ("  (dataset provided)" if DATASET else "  (dataset will be generated)"))

In [ ]:
# ---- Models to search over ------------------------------------------------
# The optimizer searches over (model x strategy). The strategies are NOT written
# here — the model designs them further down. Add "claude-opus-4-8" to include Opus.
MODELS = ["claude-haiku-4-5", "claude-sonnet-5", "claude-opus-4-8"]
print("Models:", ", ".join(MODELS))

In [ ]:
# ---- One real API call ----------------------------------------------------
def _call_api(model, prompt, max_tokens):
    # Single real API call -> (text, tokens_in, tokens_out).
    # thinking=disabled so the *strategy* is the only reasoning knob (otherwise
    # newer models reason by default, making a "direct" answer neither direct nor
    # cheap). No `temperature` (Sonnet 5 / Opus 4.8 reject it); sampling diversity
    # for majority-vote strategies comes from the model's default sampling.
    msg = client.messages.create(
        model=model, max_tokens=max_tokens,
        thinking={"type": "disabled"},
        messages=[{"role": "user", "content": prompt}],
    )
    text = "".join(b.text for b in msg.content if b.type == "text")
    return text, msg.usage.input_tokens, msg.usage.output_tokens

In [ ]:
# ---- Pull the answer out of text, and check it against the gold answer -----
# The task profile (next cell) picks the types:
#   extract_type in {"last_number", "last_line", "full"}
#   check_type   in {"numeric", "exact", "llm_judge"}

def extract_number(text):
    # The last number appearing in the text, or None. Also handed to workflow
    # programs so they can parse their own intermediate results.
    numbers = re.findall(r"-?\d[\d,]*\.?\d*", text or "")
    if not numbers:
        return None
    try:
        return float(numbers[-1].replace(",", ""))
    except ValueError:
        return None

def extract_answer(text, extract_type):
    if extract_type == "last_number":
        number = extract_number(text)
        if number is None:
            return ""
        return str(int(number)) if number == int(number) else str(number)
    if extract_type == "last_line":
        last = ""
        for line in (text or "").splitlines():
            if line.strip():
                last = line.strip()
        return last
    return (text or "").strip()          # "full": the whole thing

def check_answer(prediction, gold, check_type):
    if check_type == "numeric":
        p = extract_number(str(prediction))
        g = extract_number(str(gold))
        return p is not None and g is not None and abs(p - g) < 1e-6
    if check_type == "exact":
        return str(prediction).strip().casefold() == str(gold).strip().casefold()
    # "llm_judge": ask a cheap model whether the prediction matches the gold answer.
    prompt = (f"Reference answer: {gold}\nCandidate answer: {prediction}\n"
              "Is the candidate correct / equivalent to the reference? Answer yes or no.")
    reply, _, _ = _call_api("claude-haiku-4-5", prompt, 16)
    return reply.strip().lower().startswith("y")

In [ ]:
# ---- Profile the task: infer the answer format, and get a dataset ----------
PROFILER_MODEL = "claude-opus-4-8"

def ask_for_json(model, prompt, schema):
    # One structured-output call: the reply is guaranteed to match `schema`.
    message = client.messages.create(
        model=model, max_tokens=2048, thinking={"type": "disabled"},
        output_config={"format": {"type": "json_schema", "schema": schema}},
        messages=[{"role": "user", "content": prompt}])
    text = ""
    for block in message.content:
        if block.type == "text":
            text += block.text
    return json.loads(text)

PROFILE_SCHEMA = {
    "type": "object",
    "properties": {
        "description":  {"type": "string"},
        "answer_type":  {"type": "string"},
        "extract_type": {"type": "string", "enum": ["last_number", "last_line", "full"]},
        "check_type":   {"type": "string", "enum": ["numeric", "exact", "llm_judge"]},
    },
    "required": ["description", "answer_type", "extract_type", "check_type"],
    "additionalProperties": False,
}

def profile_task(seed_prompt, dataset):
    examples = ""
    if dataset:
        examples = "\n\nExamples (input -> answer):\n"
        for item in dataset[:5]:
            examples += f"- {item['question']} -> {item['answer']}\n"
    prompt = (
        f"A user wants to build an LLM workflow for this task:\n{seed_prompt}{examples}\n\n"
        "Reply with:\n"
        "- description: one clear paragraph describing the task for a workflow designer.\n"
        "- answer_type: a short label (e.g. 'integer', 'category', 'free text').\n"
        "- extract_type: how to pull the answer from a model's raw text — 'last_number' "
        "for numbers, 'last_line' for a short label on its own line, else 'full'.\n"
        "- check_type: how to score it — 'numeric' for numbers, 'exact' for short "
        "labels/strings, 'llm_judge' for free-form or semantic answers.")
    return ask_for_json(PROFILER_MODEL, prompt, PROFILE_SCHEMA)

DATASET_SCHEMA = {
    "type": "object",
    "properties": {"data": {"type": "array", "items": {
        "type": "object",
        "properties": {"question": {"type": "string"}, "answer": {"type": "string"}},
        "required": ["question", "answer"], "additionalProperties": False}}},
    "required": ["data"], "additionalProperties": False,
}

def generate_dataset(description, n=12):
    prompt = (
        f"Generate {n} diverse labeled examples for this task:\n{description}\n\n"
        "Each 'answer' must be the correct final target ONLY — a bare value (the number "
        "or the label), with no explanation or units. Make them solvable and unambiguous.")
    return ask_for_json(PROFILER_MODEL, prompt, DATASET_SCHEMA)["data"]

profile = profile_task(SEED_PROMPT, DATASET)
EXTRACT_TYPE = profile["extract_type"]
CHECK_TYPE   = profile["check_type"]

if DATASET is not None:
    DATA = DATASET
else:
    DATA = generate_dataset(profile["description"])

# Split: the designer may self-test on DEV; the final ranking is on held-out TEST.
split = max(1, len(DATA) * 3 // 5)
DATA_DEV = DATA[:split]
DATA_TEST = DATA[split:]

print(f"answer_type = {profile['answer_type']}   extract = {EXTRACT_TYPE}   check = {CHECK_TYPE}")
print(f"{len(DATA)} examples  ->  {len(DATA_DEV)} dev / {len(DATA_TEST)} test")
print("Description:", profile["description"])

In [ ]:
# ---- Pareto frontier + picking a workflow under a constraint --------------
def cost_of(result):
    return result["cost_per_query"]

def pareto_front(results):
    # Keep a result only if no OTHER result is at least as accurate AND at least
    # as cheap (and strictly better on at least one of the two).
    front = []
    for r in results:
        dominated = False
        for other in results:
            if other is r:
                continue
            at_least_as_cheap    = other["cost_per_query"] <= r["cost_per_query"]
            at_least_as_accurate = other["accuracy"] >= r["accuracy"]
            strictly_better      = (other["cost_per_query"] < r["cost_per_query"]
                                    or other["accuracy"] > r["accuracy"])
            if at_least_as_cheap and at_least_as_accurate and strictly_better:
                dominated = True
                break
        if not dominated:
            front.append(r)
    front.sort(key=cost_of)          # cheapest first
    return front

def best_under_budget(results, max_cost):
    best = None
    for r in results:
        if r["cost_per_query"] <= max_cost:
            if best is None or r["accuracy"] > best["accuracy"]:
                best = r
    return best

def cheapest_above_accuracy(results, min_accuracy):
    cheapest = None
    for r in results:
        if r["accuracy"] >= min_accuracy:
            if cheapest is None or r["cost_per_query"] < cheapest["cost_per_query"]:
                cheapest = r
    return cheapest

## Design workflows: a Claude Code agent writes + tests candidate programs

The proposer is a **Claude Agent SDK** agent with web search + Bash + file tools.
Its method and its dev evaluator are two standard `SKILL.md` skills under `skills/`:

- **workflow-design** — how to design candidates + the `solve(question, llm)` contract.
- **workflow-eval** — a bundled `eval_candidate.py` that scores a candidate on the dev set.

We copy both into the agent's `.claude/skills/` so the SDK discovers them, run the
agent in a **separate process** (`run_proposer.py`), and read back the programs it
picked. Each candidate still runs through our metered, budget-capped runtime — on
DEV during design, and on the held-out TEST split in the search below.

> Requires `claude-agent-sdk` + `ANTHROPIC_API_KEY`; the agent's tool calls and its
> self-tests make real API calls (spends money).

In [ ]:
# ---- Run the design agent, then read back the programs it selected ---------
# The agent runs in a subprocess (run_proposer.py) so its async event loop is
# isolated from the notebook kernel. It reads its config + the two skills from a
# scratch directory and writes its chosen programs to programs.json there.
import tempfile, pathlib, shutil, subprocess, sys

PROPOSER_MODEL = "claude-opus-4-8"

AGENT_DIR = pathlib.Path(tempfile.mkdtemp(prefix="proposer_"))
(AGENT_DIR / "task_spec.json").write_text(json.dumps(profile))
(AGENT_DIR / "dev_task.json").write_text(json.dumps(DATA_DEV[:5]))   # a few dev examples

# Copy the two skills into the agent's project directory.
skills_dir = AGENT_DIR / ".claude" / "skills"
skills_dir.mkdir(parents=True)
for skill_name in ("workflow-design", "workflow-eval"):
    shutil.copytree(pathlib.Path("skills") / skill_name, skills_dir / skill_name)

agent_prompt = (
    "Design inference-time LLM workflows for this task:\n"
    + profile["description"] + "\n\n"
    "Available models, cheap -> expensive: " + ", ".join(MODELS) + ".\n"
    "Answers are scored by extract='" + EXTRACT_TYPE + "' then check='" + CHECK_TYPE + "'.\n\n"
    "Use the workflow-design skill for the method and the program contract, and the "
    "workflow-eval skill to test each candidate on the dev set. Keep 4-5 diverse, "
    "working candidates spanning cheap -> accurate, then write programs.json and stop."
)
(AGENT_DIR / "proposer_config.json").write_text(json.dumps({
    "model": PROPOSER_MODEL,
    "skills": ["workflow-design", "workflow-eval"],
    "allowed_tools": ["WebSearch", "WebFetch", "Bash", "Read", "Write", "Skill"],
    "prompt": agent_prompt,
}))

# Run the agent in a subprocess and stream its log live.
runner = str(pathlib.Path("run_proposer.py").resolve())
process = subprocess.Popen([sys.executable, runner], cwd=str(AGENT_DIR),
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True, bufsize=1)
for line in process.stdout:
    print(line, end="")
process.wait()

# Read back the programs the agent selected.
programs_file = AGENT_DIR / "programs.json"
if programs_file.exists():
    data = json.loads(programs_file.read_text())
    PROGRAMS = data["programs"] if isinstance(data, dict) else data
else:
    # The agent didn't write programs.json (e.g. a dropped connection) — recover
    # the candidate .py files it left on disk instead.
    print(f"\nprograms.json missing — recovering candidate files from {AGENT_DIR}")
    PROGRAMS = []
    for f in sorted(AGENT_DIR.glob("*.py")):
        if f.name in ("run_proposer.py", "eval_candidate.py"):
            continue
        code = f.read_text()
        if "def solve" in code:
            PROGRAMS.append({"name": f.stem, "description": "(recovered)", "code": code})

if not PROGRAMS:
    raise RuntimeError(f"the agent produced no candidate programs (see {AGENT_DIR})")

print(f"\n{len(PROGRAMS)} programs proposed:")
for program in PROGRAMS:
    print(f"  {program['name']:22s} — {program.get('description', '')}")

In [ ]:
# ---- Run a workflow program, metering + capping every model call ----------
from collections import Counter

class Runtime:
    """A workflow program calls `runtime.llm(...)` for every model call. This is
    the one place cost is measured, and it stops the program if it goes over budget."""
    def __init__(self, default_model, max_calls=24, max_tokens=120_000):
        self.default_model = default_model
        self.max_calls = max_calls
        self.max_tokens = max_tokens
        self.calls = 0
        self.tokens = 0
        self.cost = 0.0

    def llm(self, prompt, max_tokens=256, model=None):
        if self.calls >= self.max_calls:
            raise RuntimeError("workflow exceeded its model-call budget")
        self.calls += 1
        if model not in PRICES:
            model = self.default_model
        text, tokens_in, tokens_out = _call_api(model, str(prompt), int(max_tokens))
        self.tokens += tokens_in + tokens_out
        self.cost += cost_usd(model, tokens_in, tokens_out)
        if self.tokens > self.max_tokens:
            raise RuntimeError("workflow exceeded its token budget")
        return text

def compile_solve(code):
    # Run the program's source and return its solve() function. The program may
    # use these names without importing them. (Model-written code runs with full
    # Python here — fine for a trusted research notebook; sandbox it for untrusted code.)
    namespace = {"re": re, "json": json, "statistics": statistics,
                 "Counter": Counter, "extract_number": extract_number, "MODELS": MODELS}
    exec(code, namespace)
    return namespace["solve"]

def evaluate_program(program, dataset, default_model):
    try:
        solve = compile_solve(program["code"])
    except Exception:
        return {"name": program["name"], "accuracy": 0.0, "cost_per_query": 0.0}

    n_correct = 0
    total_cost = 0.0
    for item in dataset:
        runtime = Runtime(default_model)
        try:
            raw = solve(item["question"], runtime.llm)
            prediction = extract_answer(str(raw), EXTRACT_TYPE)
            if check_answer(prediction, item["answer"], CHECK_TYPE):
                n_correct += 1
        except Exception:
            pass                     # over budget, or a bug in the program -> counts as wrong
        total_cost += runtime.cost

    return {"name": program["name"],
            "accuracy": n_correct / len(dataset),
            "cost_per_query": total_cost / len(dataset)}

In [ ]:
# ---- Run every proposed program on the held-out TEST split ----------------
# The programs are arbitrary code, so we can't estimate cost up front — we just
# run each one and measure. The per-query budget caps bound the worst case.
BASE_MODEL = MODELS[0]        # a program's default model; it may escalate to a pricier one

code_results = []
for program in PROGRAMS:
    result = evaluate_program(program, DATA_TEST, BASE_MODEL)
    print(f"  {result['name']:22s}  accuracy {result['accuracy']:.2f}   ${result['cost_per_query']:.5f}/query")
    code_results.append(result)

pd.DataFrame(code_results).sort_values(
    ["accuracy", "cost_per_query"], ascending=[False, True]).reset_index(drop=True)

In [ ]:
# ---- The Pareto frontier, the two constrained picks, and a plot ------------
frontier = pareto_front(code_results)

print("Pareto frontier (cheapest -> most accurate):")
for r in frontier:
    print(f"  {r['name']:22s}  accuracy {r['accuracy']:.2f}   ${r['cost_per_query']:.5f}/query")

BUDGET = 0.002
best = best_under_budget(code_results, BUDGET)
if best is None:
    print(f"\nNo program costs under ${BUDGET}/query.")
else:
    print(f"\nBest program under ${BUDGET}/query:  {best['name']}"
          f"  (accuracy {best['accuracy']:.2f}, ${best['cost_per_query']:.5f})")

TARGET = 0.80
cheapest = cheapest_above_accuracy(code_results, TARGET)
if cheapest is None:
    print(f"No program reaches accuracy {TARGET}.")
else:
    print(f"Cheapest program with accuracy >= {TARGET}:  {cheapest['name']}"
          f"  (accuracy {cheapest['accuracy']:.2f}, ${cheapest['cost_per_query']:.5f})")

# Plot every program as a point; draw a line through the frontier.
frontier_costs = []
frontier_accuracies = []
for r in frontier:
    frontier_costs.append(r["cost_per_query"])
    frontier_accuracies.append(r["accuracy"])

plt.figure(figsize=(7, 5))
for r in code_results:
    plt.scatter(r["cost_per_query"], r["accuracy"], color="#888888")
    plt.annotate(r["name"], (r["cost_per_query"], r["accuracy"]),
                 fontsize=8, xytext=(5, 4), textcoords="offset points")
plt.plot(frontier_costs, frontier_accuracies, "-o", color="#d9534f", label="Pareto frontier")
plt.xlabel("cost per query (USD)")
plt.ylabel("accuracy")
plt.title("LLM-designed workflows: accuracy vs. cost")
plt.legend()
plt.tight_layout()
plt.show()